In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import torch
from darts.utils.statistics import plot_ccf, remove_seasonality, granger_causality_tests
from darts.utils.statistics import stationarity_test_adf
from darts.utils.utils import SeasonalityMode
from sklearn.preprocessing import StandardScaler

from aare.constants import TIME
from aare.params import read_params
from aare.preparation import resample, interpolate
from aare.remote_existenz_store import RemoteExistenzStore
from aare.utils import to_ts

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
# logging.basicConfig(level="DEBUG")

In [ ]:
params = read_params()
store = RemoteExistenzStore()

# Variables to look at

<https://api-datasette.konzept.space/existenz-api/hydro_parameters> \
<https://api-datasette.konzept.space/existenz-api/smn_parameters>

- Water temperature from Thun (hydro/temperature):
  - Thun is upstream from Bern, so how long do changes in the water temperature there take to come into effect in Bern? Are they negligible?
- Flow (hydro/flow):
  - Does the flow or change in flow have an influence on the temperature or change in temperature?
- Precipitation (smn/rr):
  - Does the precipitation have an influence on the temperature or change in temperature?
  - Does the precipitation have an influence on the flow? Probably yes, but at which lag is the correlation greatest?
  - To be useful as feature, pool together a total over some hours and make sure the model has access to all relevant lags.
- Sunshine duration (smn/ss):
  - Does the sunshine duration have an influence on the temperature or change in temperature?
  - How much lag is there in the heat transfer from the sunshine to the water temperature? could influence water directly without air temperature
- Air temperature (smn/tt):
  - How high is the correlation between air temperature and water temperature?
  - How much lag is there in the heat transfer from the air temperature to the water temperature?
  - Does the daily, weekly or monthly mean show correlation to the change in water temperature? might be nice feature
- Turbidity (hydro/turbidity):
  - Might influence the transfer rate from sunshine to water temperature
- Global Exposure (smn/rad):
  - Don't quite know what this is and does.
  - Probably just affects air temperature and won't make a good feature but idk.
- Relative Humidity (smn/rh):
  - Does humidity have any direct influence on the water temperature? If there is correlation, I suspect it's just because air temperature and precipitation affect RH.
  - Probably not a good feature

 Unsure of confounders:
 - Precipitation affects flow
 - Sunshine affects air temperature
 - Turbidity is affected by flow and precipitation

### Note on lags

If these features are to be used for forecasting, you must make sure that either

1. their influence delay is larger than our forecast horizon (if it's delayed > 4 days, can use data from now to predict 4 days into the future)
1. there is a forecast available for the data (we can use existing/official forecasts for air temperature, precipitation and flow to help forecasts further out)
1. we are also forecasting that variable to be able to use it autoregressively for future forecasts

If none of these are true, we cannot use it to forecast for the desired horizon and need to shorten, drop the feature or forecast it ourselves (bullet 3).

### Note on frequency

For some features, like the precipitation, we probably don't want the average of 10 min totals over an hour (agg mean) but rather the total over an hour (agg sum).
Same goes for sunshine duration for example.

### Note on availability

Not all of those features might be available as far back as the water temperature.
If we find that some feature would be really nice for training, but it's only available for the last few years,
we might need to consider using less data for training and validation. If performance is not good enough, could think about pre-training.

In [ ]:
ANYTIME = "0"  # to be used as period start when querying influx. starting at 0 just returns all the data.

In [ ]:
df = store.query(
    ANYTIME,
    [
        "hydro/temperature:mean_6h@bern",
        "hydro/flow:mean_6h@bern",
        "hydro/flow:mean_6h@thun",
        "smn/rr:sum_6h@bern",
        "smn/ss:sum_6h@bern",
        "smn/rad:sum_6h@bern",
        "smn/tt:mean_6h@bern",
        "smn/rh:sum_6h@bern",
    ],
)
df

In [ ]:
px.scatter(df, x=TIME, y=[c for c in df.columns if c != TIME])

In [ ]:
df = store.query(
    ("2024-01-01", "2025-01-01"),
    [
        "hydro/temperature:mean_1h@bern",
        "hydro/temperature:mean_1h@thun",
        "hydro/flow:mean_1h@bern",
        "hydro/flow:mean_1h@thun",
        "smn/rr:sum_1h@bern",
        "smn/rr:sum_1h@thun",
        "smn/ss:sum_1h@bern",
        "smn/ss:sum_1h@thun",
        "smn/rad:sum_1h@bern",
        "smn/rad:sum_1h@thun",
        # "hydro/turbidity:mean_1h@bern" <- not returned apparently? must investigate
        "smn/tt:mean_1h@bern",
        "smn/tt:mean_1h@thun",
    ],
)
df

In [ ]:
df = resample(df)
df

In [ ]:
df.isna().sum()

In [ ]:
df = interpolate(df, drop_filled=True, columns=None)
df

In [ ]:
df.isna().sum()

In [ ]:
scaler = StandardScaler()
df_with_index = df.set_index(TIME)
scaled_values = scaler.fit_transform(df_with_index.to_numpy(copy=False))
df_s = pd.DataFrame(scaled_values, index=df_with_index.index, columns=df_with_index.columns).reset_index()

In [ ]:
df.describe()

In [ ]:
df_s.describe()

## Temperature Thun -> Temperature Bern

In [ ]:
px.scatter(df, x=TIME, y=["temperature_bern", "temperature_thun"])

In [ ]:
temp_bern = to_ts(df, col="temperature_bern")
temp_thun = to_ts(df, col="temperature_thun")

In [ ]:
plot_ccf(temp_bern, temp_thun, max_lag=4 * 24)

In [ ]:
plot_ccf(remove_seasonality(temp_bern, freq=24), remove_seasonality(temp_thun, freq=24), max_lag=4 * 24)

In [ ]:
plot_ccf(
    remove_seasonality(temp_bern, freq=24, method="STL", model=SeasonalityMode.ADDITIVE),
    remove_seasonality(temp_thun, freq=24, method="STL", model=SeasonalityMode.ADDITIVE),
    max_lag=4 * 24,
)

Over a long period, the correlation between temp THUN and BERN is very high everywhere, even if we remove the daily seasonality.
But we can also check if the change in THUN has a correlation with the change in BERN, since that is a stationary series.

In [ ]:
plot_ccf(temp_bern.diff(), temp_thun.diff(), max_lag=4 * 24)

In [ ]:
plot_ccf(remove_seasonality(temp_bern, freq=24).diff(), remove_seasonality(temp_thun, freq=24).diff(), max_lag=4 * 24)

In [ ]:
# switching the places shows that temp_thun barely has any correlation with lagged values of temp_bern, meaning
# temp_thun is leading/ahead of temp_bern (see bern lag 0 is correlated with thun lag 4).
plot_ccf(remove_seasonality(temp_thun, freq=24).diff(), remove_seasonality(temp_bern, freq=24).diff(), max_lag=4 * 24)

And indeed, it seems that there is a peak correlation at around 3-4 hours, meaning when the temperature changes in THUN, it will likely also change in BERN 3-4 hours later.

IMPORTANT EDIT: Technically this means that a positive change in the _rate of change_ of the temperature in THUN will likely lead to a positive change in _rate of change_ of the temperature in BERN 3-4 hours later. I think this can be interpreted as: sudden/quick changes in THUN will lead to sudden/quick changes in BERN a bit later. Given the domain, this can probably be traced back to the more naive interpretation, which would be applicable if we didn't difference the series.

The lagged temperature (change) from THUN might therefore be an interesting feature for forecasting the temperature in BERN.
We can try to reinforce that with Granger Causality, but note that this test is primarily used for economics IIRC.

In [ ]:
# see p-value (second value) is > 0.05 -> likely non-stationary
stationarity_test_adf(temp_bern)

In [ ]:
# see p-value (second value) is << 0.05 -> very likely stationary
stationarity_test_adf(temp_bern.diff())

In [ ]:
# noinspection PyNoneFunctionAssignment
gc_report = granger_causality_tests(temp_thun.diff(), temp_bern.diff(), maxlag=8)

In [ ]:
gc_report

In [ ]:
# Print for all lags, whether the null hypothesis is accepted.
# The null hypothesis is: temp_thun does NOT granger-cause temp_bern, meaning False = there could be causality.
{k: all(s[1] > 0.05 for s in v[0].values()) for k, v in gc_report.items()}

Although this Granger Causality Test clearly rejects the null hypothesis, it's still possible that the alternative hypothesis (temp_thun causes temp_bern) is wrong,
because there is 1) a very high correlation between the two and 2) there are many confounders that influence both temp_thun and temp_bern.
But then again, it's also possible that temp_thun is a useful feature for forecasting temp_bern, even if there were none or barely any causality (if we want to use that word).

Interestingly, the p-values for lag 1 are highest, then lag 2, meaning it's less likely that lag 1 thun is causing bern than lag 3 thun causing bern.
But they are so incredibly low that I don't think we should overthink that.
It does kinda make sense though, because the lowest lags are the furthers away from having any influence, the water simply isn't there yet.

## Flow Bern -> Temperature Bern

In [ ]:
px.scatter(df, x=TIME, y=["temperature_bern", "flow_bern"])

In [ ]:
px.scatter(df_s, x=TIME, y=["temperature_bern", "flow_bern"], title="temp_bern and flow_bern normalized")

In [ ]:
temp_bern_s = to_ts(df_s, col="temperature_bern")
flow_bern = to_ts(df, col="flow_bern")
flow_bern_s = to_ts(df_s, col="flow_bern")

In [ ]:
plot_ccf(temp_bern, flow_bern, max_lag=4 * 24)

In [ ]:
# for ccf, it does not matter whether the data is normalized or not
plot_ccf(temp_bern_s, flow_bern_s, max_lag=4 * 24)

In [ ]:
plot_ccf(temp_bern.diff(), flow_bern.diff(), max_lag=4 * 24)

In [ ]:
plot_ccf(flow_bern.diff(), temp_bern.diff(), max_lag=4 * 24)

We can observe a significant negative correlation at lag 1 for DIFF FLOW -> DIFF TEMPERATURE. Switching the positions shows that flow is leading the temperature because lagging the flow results in higher correlations that lagging the temperature.


~~I would interpret this as: when the flow increases, an hour later the temperature will decrease, which I think might make sense.~~ \
I would interpret this as: when the increase of the flow goes up (high rate of change, e.g. sudden increase), then the rate of change of the temperature _decreases_. This could mean that temperatures either stabilizes or _turns around_ (if it was going up steadily and more water is added, that increase could be slowed because more water needs to be heated).

This means the flow at lag 1 might be an interesting feature to forecast the temperature, but it's also possible that this correlation is already modeled by other relationships with confounding variables (i.e. if precipitation has a negative correlation with temperature and also a positive correlation with the flow, it might not be necessary to include both variables).


In [ ]:
plot_ccf(abs(temp_bern.diff()), abs(flow_bern.diff()), max_lag=4 * 24)

In [ ]:
# FYI: same result whether you remove seasonality before or after diffing
plot_ccf(
    remove_seasonality(temp_bern.diff(), freq=24, model=SeasonalityMode.ADDITIVE), flow_bern.diff(), max_lag=4 * 24
)

Looking at the absolute rates of change and the NCC after trying to remove the daily seasonality still present in the temp diffs, it doesn't confirm or deny anything. Maybe it shows that it's not as significant of a correlation as others.

## Precipitation -> Temperature and Flow?

We expect that there is a lagged correlation between precipitation and flow, let's confirm that.

In [ ]:
px.scatter(df, x=TIME, y=["rr_bern", "flow_bern"], title="precipitation_bern and flow_bern")

In [ ]:
px.scatter(df_s, x=TIME, y=["rr_bern", "flow_bern"], title="precipitation_bern and flow_bern normalized")

In [ ]:
rr_bern = to_ts(df, col="rr_bern")

In [ ]:
plot_ccf(flow_bern, rr_bern, max_lag=4 * 24)

In [ ]:
plot_ccf(flow_bern.diff(), rr_bern.diff(), max_lag=4 * 24)

Visually, there seems to be a correlation after a few hours, but the CCF plot shows that the correlation is highest at 11 hours of lag. Given the non-stationarity of these series, we also look at the CCF of the diff'd series. This shows a peak at around 6 hours, so fast changes in precipitation lead to fast changes in flow around 6 hours later.

We're not predicting the flow anyway, but if we were, it seems that values from the last 24h could be helpful. Might also want to analyze aggregate (e.g. daily sum).

But now let's do correlation with temperature.

In [ ]:
px.scatter(df, x=TIME, y=["temperature_bern", "rr_bern"])

In [ ]:
px.scatter(df_s, x=TIME, y=["temperature_bern", "rr_bern"])

Visually, I can't make out any significant correlation.

In [ ]:
plot_ccf(temp_bern, rr_bern, max_lag=4 * 24)

In [ ]:
plot_ccf(temp_bern.diff(), rr_bern.diff(), max_lag=4 * 24)

And the CCF agree strongly. Therefore, we should probably not include precipitation to forecast the temperature.

## Air temperature -> Water temperature

We expect this to be one of the strongest contributors to the variable.

In [ ]:
px.scatter(df, x=TIME, y=["temperature_bern", "tt_bern"])

Visually, we can already see a significant correlation, but they both have the same underlying seasonality.

In [ ]:
tt_bern = to_ts(df, col="tt_bern")

In [ ]:
plot_ccf(temp_bern, tt_bern, max_lag=4 * 24)

In [ ]:
plot_ccf(temp_bern.diff(), tt_bern.diff(), max_lag=4 * 24)

In [ ]:
plot_ccf(
    remove_seasonality(temp_bern, 24, model=SeasonalityMode.ADDITIVE),
    remove_seasonality(tt_bern, 24, model=SeasonalityMode.ADDITIVE),
    max_lag=4 * 24,
)

In [ ]:
plot_ccf(
    remove_seasonality(temp_bern, 24, model=SeasonalityMode.ADDITIVE).diff(),
    remove_seasonality(tt_bern, 24, model=SeasonalityMode.ADDITIVE).diff(),
    max_lag=4 * 24,
)

Must remove strong daily seasonality and differentiate to get a useful CCF, and even then you can see residual seasonality.

Still, it's clear that there is a strong correlation between air temperature and water temperature, already at the early lags, but it seems to peak at 1 hour lag.

The air temperature is definitely a good feature to include when forecasting water temperature. Let's also check the same thing for Thun.

In [ ]:
px.scatter(df, x=TIME, y=["temperature_thun", "tt_thun"])

In [ ]:
tt_thun = to_ts(df, col="tt_thun")

In [ ]:
plot_ccf(
    remove_seasonality(temp_thun, 24, model=SeasonalityMode.ADDITIVE).diff(),
    remove_seasonality(tt_thun, 24, model=SeasonalityMode.ADDITIVE).diff(),
    max_lag=4 * 24,
)

And from thun to bern

In [ ]:
px.scatter(df, x=TIME, y=["temperature_bern", "tt_thun"])

In [ ]:
plot_ccf(
    remove_seasonality(temp_bern, 24, model=SeasonalityMode.ADDITIVE).diff(),
    remove_seasonality(tt_thun, 24, model=SeasonalityMode.ADDITIVE).diff(),
    max_lag=4 * 24,
)

We can also look at the isolated effect of thun vs bern. Usually you would de-confound the variable with a regression, but we can also just look at the correlation between the difference of thun-bern air and bern water. Aka if its X degrees hotter in thun than in bern, what effect does this have on the water temperature? Intuitively, this should result in higher lags with peak correlation because the air temp has to go into the water in thun and then propagate downstream. This isn't the case usually because generally the air temperature in bern is highly correlated with the one in bern.

This shows that temperature changes in thun take a longer time to affect the water temperature in bern, as expected.

In [ ]:
px.scatter(
    x=df[TIME], y=[df["temperature_bern"], df["temperature_thun"], df["temperature_thun"] - df["temperature_bern"]]
)

In [ ]:
plot_ccf(
    remove_seasonality(temp_bern, 24, model=SeasonalityMode.ADDITIVE).diff(),
    (
        remove_seasonality(tt_thun, 24, model=SeasonalityMode.ADDITIVE)
        - remove_seasonality(tt_bern, 24, model=SeasonalityMode.ADDITIVE)
    ).diff(),
    max_lag=4 * 24,
)

This won't be possible with simple subtraction, but we could also try to isolate the effect of the air temperature without the upstream water temperature, which should be much lower.

Direct effect of thun air on bern water should be low. For direct effect we need to remove indirect effect of thun water. Since there is still a high correlation between thun air and bern air, I still expect significant correlation, despite no direct causation.

In [ ]:
temp_thun_s = to_ts(df_s, col="temperature_thun")
tt_thun_s = to_ts(df_s, col="tt_thun")
tt_bern_s = to_ts(df_s, col="tt_bern")

In [ ]:
val_temp_bern = remove_seasonality(temp_bern, 24, model=SeasonalityMode.ADDITIVE).values().ravel()
val_temp_thun = remove_seasonality(temp_thun, 24, model=SeasonalityMode.ADDITIVE).values().ravel()

In [ ]:
# https://www.phind.com/search/cm7vn78t80000206isw2x5i88
poly = np.polynomial.Polynomial.fit(val_temp_thun, val_temp_bern, 1)
poly

In [ ]:
val_temp_bern_deconfound = val_temp_bern - poly(val_temp_thun)
val_temp_bern_deconfound

In [ ]:
# turns out, in this case it's almost the same as just bern - thun
px.scatter(x=df[TIME], y=[val_temp_bern, val_temp_thun, val_temp_bern_deconfound, val_temp_bern - val_temp_thun])

In [ ]:
from darts import TimeSeries

temp_bern_deconfound = TimeSeries.from_times_and_values(temp_bern.time_index, val_temp_bern_deconfound)

In [ ]:
plot_ccf(
    temp_bern_deconfound.diff(),
    remove_seasonality(tt_thun_s, 24, model=SeasonalityMode.ADDITIVE).diff(),
    max_lag=4 * 24,
)

In [ ]:
# here the air temp thun to water temp bern again without isolation
plot_ccf(
    remove_seasonality(temp_bern_s, 24, model=SeasonalityMode.ADDITIVE).diff(),
    remove_seasonality(tt_thun_s, 24, model=SeasonalityMode.ADDITIVE).diff(),
    max_lag=4 * 24,
)

In [ ]:
# here the water temp thun to water temp bern again without isolation
plot_ccf(
    remove_seasonality(temp_bern_s, 24, model=SeasonalityMode.ADDITIVE).diff(),
    remove_seasonality(temp_thun_s, 24, model=SeasonalityMode.ADDITIVE).diff(),
    max_lag=4 * 24,
)

We can see the correlation decreases, and it also shows a higher delay. I'm not 100% sure how to interpret this, but I guess it does make sense: Changes in the air temperature in thun can coincide with changes in the water temperature in bern, even if they are not propagated via the water (again, thun and bern air temp are highly correlated) but by air for example. Also the deconfounding might not be perfect.

## Sunshine duration -> Water temperature (and air temperature?)

It would be interesting to see if the sunshine duration helps IN ADDITION to the air temperature, which will be correlated.

In [ ]:
px.scatter(df, x=TIME, y=["temperature_bern", "ss_bern"])

In [ ]:
df["ss_bern_cum"] = df["ss_bern"].groupby(df[TIME].dt.date).transform("cumsum")
df["ss_thun_cum"] = df["ss_thun"].groupby(df[TIME].dt.date).transform("cumsum")
df_s["ss_bern_cum"] = df_s["ss_bern"].groupby(df_s[TIME].dt.date).transform("cumsum")
df_s["ss_thun_cum"] = df_s["ss_thun"].groupby(df_s[TIME].dt.date).transform("cumsum")

In [ ]:
px.scatter(df_s, x=TIME, y=["temperature_bern", "ss_bern", "ss_bern_cum", "tt_bern"])

In [ ]:
ss_bern = to_ts(df, col="ss_bern")

In [ ]:
plot_ccf(temp_bern, ss_bern, max_lag=4 * 24)

In [ ]:
plot_ccf(temp_bern.diff(), ss_bern.diff(), max_lag=4 * 24)

In [ ]:
plot_ccf(
    remove_seasonality(temp_bern, 24, model=SeasonalityMode.ADDITIVE),
    remove_seasonality(ss_bern, 24, model=SeasonalityMode.ADDITIVE),
    max_lag=4 * 24,
)

In [ ]:
plot_ccf(
    remove_seasonality(temp_bern, 24, model=SeasonalityMode.ADDITIVE).diff(),
    remove_seasonality(ss_bern, 24, model=SeasonalityMode.ADDITIVE).diff(),
    max_lag=4 * 24,
)

In [ ]:
ss_bern_cum = to_ts(df, col="ss_bern_cum")

In [ ]:
plot_ccf(temp_bern, ss_bern_cum, max_lag=4 * 24)

In [ ]:
plot_ccf(temp_bern.diff(), ss_bern_cum.diff(), max_lag=4 * 24)

In [ ]:
plot_ccf(
    remove_seasonality(temp_bern, 24, model=SeasonalityMode.ADDITIVE),
    remove_seasonality(ss_bern_cum, 24, model=SeasonalityMode.ADDITIVE),
    max_lag=4 * 24,
)

In [ ]:
plot_ccf(
    remove_seasonality(temp_bern, 24, model=SeasonalityMode.ADDITIVE).diff(),
    remove_seasonality(ss_bern_cum, 24, model=SeasonalityMode.ADDITIVE).diff(),
    max_lag=4 * 24,
)

The cumulative says: how much sunshine has already happened today. Maybe it would be more interesting to know, how much sunshine happened in the last 6 hours or so.

In [ ]:
cum_tests = range(2, 14)
for i in cum_tests:
    df[f"ss_bern_cum{i}"] = df["ss_bern"].rolling(i, min_periods=1).sum()
    df_s[f"ss_bern_cum{i}"] = df_s["ss_bern"].rolling(i, min_periods=1).sum()

In [ ]:
px.scatter(df_s, x=TIME, y=["temperature_bern", "ss_bern", "ss_bern_cum2", "ss_bern_cum4", "ss_bern_cum6", "tt_bern"])

In [ ]:
cums = [to_ts(df, col=f"ss_bern_cum{i}") for i in cum_tests]

In [ ]:
for ts in [ss_bern] + cums:
    plot_ccf(
        remove_seasonality(temp_bern, 24, model=SeasonalityMode.ADDITIVE).diff(),
        remove_seasonality(ts, 24, model=SeasonalityMode.ADDITIVE).diff(),
        max_lag=4 * 24,
    )

This analysis shows that the cumulative sunshine duration of the last 7h has the most similar curve to the water temperature = highest correlation at lag 0 in CCF. And even slightly higher is the correlation for the last 7h skipping one, so from -01:00 to -08:00, or LAG 1 in CCF.

Looking at the plot below we can see that it's mostly correlated with the air temperature (makes sense) and therefore probably only indirectly influences the water. Still the 7h cumsum and potentially even the raw value could be useful for a model to predict. However, first a correlation analysis without influence of the air temperature needs to be done (trying to find direct effect of sunshine on water temperature).


In [ ]:
px.line(df_s, x=TIME, y=["temperature_bern", "ss_bern", "ss_bern_cum7", "tt_bern"])

In [ ]:
# Deconfound bern water temp with air temperature and try to see if the sunshine duration has a correlation with that residual.
#  If not, it's most likely a redundant feature that doesn't help to predict the water temperature and can be left out.
#  Ps. Glacier melting, but idk if we can account for that directly, just let the model learn that by the months.

In [ ]:
val_tt_bern = remove_seasonality(tt_bern, 24, model=SeasonalityMode.ADDITIVE).values().ravel()
# I think the relationship between tt and temp is non-linear so let's do order >1
poly = np.polynomial.Polynomial.fit(val_tt_bern, val_temp_bern, 3)
poly

In [ ]:
val_temp_bern_deconfound = val_temp_bern - poly(val_tt_bern)
val_temp_bern_deconfound

In [ ]:
px.scatter(x=df[TIME], y=[val_temp_bern, val_tt_bern, val_temp_bern_deconfound, val_temp_bern - val_tt_bern])

In [ ]:
px.scatter(x=df[TIME], y=[val_temp_bern, val_tt_bern, val_temp_bern_deconfound, df_s["ss_bern_cum7"]])

In [ ]:
from darts import TimeSeries

temp_bern_deconfound = TimeSeries.from_times_and_values(temp_bern.time_index, val_temp_bern_deconfound)

In [ ]:
plot_ccf(
    temp_bern_deconfound.diff(),
    remove_seasonality(ss_bern, 24, model=SeasonalityMode.ADDITIVE).diff(),
    max_lag=4 * 24,
)

In [ ]:
plot_ccf(
    temp_bern.diff(),
    remove_seasonality(ss_bern, 24, model=SeasonalityMode.ADDITIVE).diff(),
    max_lag=4 * 24,
)

In [ ]:
plot_ccf(
    temp_bern.diff(),
    remove_seasonality(tt_bern, 24, model=SeasonalityMode.ADDITIVE).diff(),
    max_lag=4 * 24,
)

In [ ]:
ss_bern_cum7 = cums[cum_tests.index(7)]

In [ ]:
plot_ccf(
    temp_bern_deconfound.diff(),
    remove_seasonality(ss_bern_cum7, 24, model=SeasonalityMode.ADDITIVE).diff(),
    max_lag=4 * 24,
)

In [ ]:
plot_ccf(
    temp_bern.diff(),
    remove_seasonality(ss_bern_cum7, 24, model=SeasonalityMode.ADDITIVE).diff(),
    max_lag=4 * 24,
)

With these charts I tried to figure out if there is a direct effect of the sunshine duration on the water temperature when excluding the direct effect that the air temperature has on the water temperature (because of course the sunshine duration has an effect on the air temperature). However, it shows a strong negative correlation at lag 0, meaning high sunshine duration leads to a decrease in water temperature. Intuitively, I disagree, and I think the reason for these results is that the de-confounding is wrong; it seems to overcompensate and behaves similarly to just subtracting the higher-scale air temperature from the lower scale water-temperature. This means in the end our de-confounded series is very similar to the negative of the air temperature and of course that would show significant (negative) correlation to the sunshine duration.

As it stands here, we cannot deduce anything useful from this. I also don't want to dive much deeper into this analysis anymore.

In [ ]:
# This is obvious, but just to show: here the correlation between sunshine and air temp
plot_ccf(tt_bern, ss_bern, max_lag=4 * 24)

In [ ]:
# This is obvious, but just to show: here the correlation between sunshine and air temp
plot_ccf(tt_bern, ss_bern, max_lag=4 * 24)

In [ ]:
# diffed and without seasonality, still pretty significant but interestingly less lagged
plot_ccf(
    remove_seasonality(tt_bern, 24, model=SeasonalityMode.ADDITIVE).diff(),
    remove_seasonality(ss_bern, 24, model=SeasonalityMode.ADDITIVE).diff(),
    max_lag=4 * 24,
)

In [ ]:
# sum of sunshine in the last 7 hours
plot_ccf(tt_bern, ss_bern_cum7, max_lag=4 * 24)

## Global exposure -> Water temperature

But first check corr with sunshine duration and air temp.

In [ ]:
# I'll save this for later, I want to get to modelling now :(

# Conclusions

- Water temperature from Thun (hydro/temperature):
  - Since Thun is upstream from Bern, it seems to take around 3-4 hours for the changes in water temperature to reach Bern.
  - Availability history: Since Jun 2001
  - Availability inference: Since there is no forecast for the Thun water temperature, this variable is only useful for predictions in the following 3-4 hours -> no data further into the future
  - Verdict: If we want to improve accuracy for the predictions in the first few hours, we could incorporate it, but otherwise leave it.
- Flow (hydro/flow):
  - There is a significant negative correlation at lag 1.
  - Availability history: Since Aug 2009
  - Availability inference: If we incorporate the existing hydrological forecast of the government, we get ~4.5 days into the future.
  - Verdict: Use flow at lag 1 to forecast current water temperature.
- Precipitation (smn/rr):
  - I was not able to find any correlation between the precipitation and the water temperature. Hydrologically, I would assume the flow and the air temperature already contain everything to know that precipitation could provide.
  - Availability history: Since Sep 2013
  - Availability inference: Forecasts by MeteoSwiss (and other sources) are readily available 5+ days into the future.
  - Verdict: Don't use
- Sunshine duration (smn/ss):
  - I was not able to confirm any _direct_ correlation between sunshine duration and water temperature on _any_ lags and even with cumulative sums e.g. sum of sunshine minutes in the last 7h.
  - This is sad because intuitively, I thought the sunshine duration would definitely help since sun can shine directly onto the water. Maybe throw it at some models still or invest more time into analysis.
  - Availability history: Since Sep 2013
  - Availability inference: Forecasts by MeteoSwiss (and other sources) are readily available 5+ days into the future.
  - Verdict: Probably do not use, or definitely not at first. Could be interesting for later experiments.
- Air temperature (smn/tt):
  - Even after removing daily and yearly seasonality as well as differencing the series, there is still a significant correlation of the air temperature to the water temperature, especially at lag 1.
  - Availability history: Since Sep 2013
  - Availability inference: Forecasts by MeteoSwiss (and other sources) are readily available 5+ days into the future.
  - Verdict: Use! Probably the best feature we have.
- Turbidity (hydro/turbidity):
  - Was not able to analyze because of missing data, decided it's probably not worth it anyway.
- Global Exposure (smn/rad):
  - Didn't analyze yet, not sure if it would add more than sunshine duration.
  - But when you get to sunshine duration, we should revisit this.
  - Availability history: Since Feb 2019
  - Availability inference: No known forecast; can only be used as past covariate. Sunshine duration is probably a proxy
- Relative Humidity (smn/rh):
  - Didn't analyze yet.
  - Availability history: Since Sep 2013
  - Availability inference: No known forecast, precipitation might be a proxy.



## Cross-reference with literature (mainly DOI 10.1111/j.1365-2427.2006.01597.x)

- Many simple models just use the air temperature
- > Water temperature is generally close to the groundwater temperature at the source (e.g. in headwater streams) and increases thereafter with distance/stream order. The increase in water temperature is not linear and the rate of increase is greater for small streams than for large rivers.
- Daily (diel) and yearly (annual) cycles is highly relevant
- River heat exchange has the following components:
  - Solar radiation or net short-wave radiation
  - net long-wave radiation
  - evaporation (evaporative heat flux)
  - air temperature (flux resulting from temperature differences between river and atmosphere)
- Precipitation does/can influence the temperature but generally has a much lower contribution than the 4 above
- The dominant contribution to the temperature is apparently **solar radiation**. Then net long-wave radiation (reflected radiation??) and evaporative heat flux (both similar contributions). **The smallest component is apparently convective heat transfer** (although not negligable).
- IIRC the air/water interface is the place where most of the engery transfer happens, so not from underneath. Solar radiation, evaporation and air to water all happen there.
- For long term forecasts, decomposition is often used and short-term and annual patterns are forecast differently.
- Stratification (layering of water by temperature) is generally negligable, esp. in rivers that aren't huge.

![](https://onlinelibrary.wiley.com/cms/asset/4c3644bb-5285-4c1e-9203-29edd488a929/fwb_1597_f1.gif)

![](https://onlinelibrary.wiley.com/cms/asset/496d8a57-56d2-4ea5-9481-d37e60a118dd/fwb_1597_f4.gif)

From DOI 10.5194/hess-25-2951-2021:

> Of these tested models, the FNN model using air temperature, discharge and precipitation and, if available, radiation as inputs produces the best-performing models.

### What I take from this

- DO NOT DISCARD/NEGLECT THE SOLAR RADIATION (or sunshine duration as proxy)!!! even if initial analysis did not yield promising results, the theory suggests that its influence is larger than the one of the air temperature IIRC. At least try a model with and without it to see if it makes a difference.
- Also analyze relative humidity and maybe try a model with it, because it could influence the evaporation component, which also has a non-negligable influence on the water temperature.
- Other than that, my assumptions and ideas seem to be okay. I've never seen turbidity mentioned and I'm not sure if its influence on sunlight absorption is negligable or not, but just FYI



## What some big LLMs have to say

### GPT-4o

Forecasting river water temperature is a complex task, influenced by a variety of hydrological and meteorological variables. The key is to select variables that directly or indirectly affect the heat exchange between the river and its environment. Here's a breakdown of important ones to consider:

---

### 🌦️ **Meteorological Variables**

These influence **energy input/output** at the river's surface:

1. **Air Temperature** (daily min/max or hourly):
   - Strongly correlated with water temperature.
   - Often used as a leading predictor in models.

2. **Solar Radiation** (or sunshine duration as a proxy):
   - Drives direct heating of the river surface.

3. **Wind Speed & Direction**:
   - Affects evaporative cooling and mixing.
   - Strong wind can induce surface cooling or enhance stratification.

4. **Relative Humidity**:
   - Influences evaporation rates and thus latent heat loss.

5. **Precipitation**:
   - Cools or warms the river depending on the relative temp of the rain/snow.
   - Influences river discharge (indirect thermal effect).

6. **Cloud Cover**:
   - Modulates solar radiation; affects both daytime heating and nighttime cooling.

7. **Air Pressure**:
   - Can indirectly affect weather patterns and temperatures, but not a primary driver.

---

### 🌊 **Hydrological Variables**

These relate to **in-stream dynamics and watershed conditions**:

1. **River Discharge (Flow Rate)**:
   - Higher flows often buffer temp changes due to greater volume.
   - Low flow = higher sensitivity to ambient conditions.

2. **Upstream Water Temperature**:
   - Critical for predicting downstream temps, especially in longer rivers.

3. **Groundwater Inflow**:
   - Usually cooler and constant, stabilizes temperature, especially in summer.

4. **River Depth & Width**:
   - Controls heat storage capacity and surface-area-to-volume ratio.

5. **Water Velocity**:
   - Influences mixing and residence time (slower flow = more heat exchange).

6. **Turbidity / Albedo**:
   - Affects solar penetration into the water column.
   - Cloudier water can heat differently than clear water.

---

### 🗺️ **Other Influential Factors**

Depending on the modeling scope or resolution, also consider:

- **Riparian Shading** (vegetation cover): impacts solar input.
- **Lake or Reservoir Inputs**: large water bodies can thermally buffer downstream rivers.
- **Seasonality / Snowmelt Timing**: key in mountainous or temperate climates.

---

### ✅ Commonly Used Core Variables (Minimum Set)

If you're building a forecasting model and need a manageable input set, a good starting point includes:

- Air temperature (daily mean or hourly)
- Solar radiation (or sunshine duration)
- Discharge / flow rate
- Previous day’s water temperature
- Wind speed (optional but useful)
- Precipitation (especially for events)

---

Would you like help building a model or dataset around this? Could tailor it more depending on the region, river size, or timescale you’re forecasting (e.g., daily vs seasonal).

---

Awesome — with an **hourly resolution** and a **forecast horizon of one week**, you're entering a more dynamic regime where **diurnal cycles, short-term weather events, and lag effects** really matter. This setup is great for machine learning models, physics-informed models, or hybrid approaches. Here’s how I’d fine-tune your variable selection:

---

### 🌦️ **Meteorological Variables (Hourly Inputs)**

These should be forecasted or available with high temporal resolution for the next 7 days:

1. **Air Temperature [°C]**
   → **Most critical predictor.** Strong diurnal pattern, often leads water temp by 3–12 hours.
   ✅ Use both *instantaneous* and *lagged* values (e.g., past 6–24h).

2. **Solar Radiation [W/m²]**
   → Drives direct heating. Important to capture daily cycle and cloud cover impacts.
   ✅ If direct measurements aren’t available, use modeled radiation or proxies (e.g., sunshine duration + cloud cover).

3. **Wind Speed [m/s] & Direction**
   → Drives surface heat exchange and mixing. Sudden gusts or fronts can have short-term cooling effects.

4. **Relative Humidity [%]**
   → Affects evaporative cooling. Especially relevant during hot, dry, or stormy periods.

5. **Precipitation [mm/h]**
   → Impacts thermal regime, especially during heavy rainfall (dilution, runoff) or snowmelt.
   ✅ Include both **instantaneous** and **cumulative rainfall** over the past 6–12h.

6. **Cloud Cover [%]** or Shortwave/Longwave Radiation
   → Controls net radiation budget. More clouds = less daytime heating, less nighttime cooling.

---

### 🌊 **Hydrological Variables (Hourly or Interpolated)**

These define the river’s **thermal response capacity**:

1. **River Discharge [m³/s]**
   → Buffers temperature variability. Use current and lagged values.
   ✅ Also consider rate of change (∆Q) if rapid flow shifts happen.

2. **Water Level or Stage [m]**
   → Proxy for depth; deeper rivers warm more slowly.

3. **Upstream Water Temperature [°C]**
   → If available, very powerful for downstream forecasting.

4. **Groundwater Input / Baseflow Index**
   → If known or can be estimated, helps understand thermal inertia.

---

### 🧠 **Temporal Features (for Model Context)**

- **Hour of Day** (captures diurnal cycle)
- **Day of Week** (for human-influenced rivers)
- **Julian Day or DOY** (captures seasonal cycle)
- **Lagged Water Temp (1h, 6h, 24h ago)**
  → Essential for autoregressive learning.

---

### 🔁 Optional / Advanced

- **Turbidity** or **Suspended Solids**: Affects heat absorption.
- **Riparian Shade Index / Vegetation**: For smaller rivers, may matter at hourly scale.
- **Snowmelt / SWE**: For mountainous basins or spring modeling.

---

### ✅ RECOMMENDED INPUT SET FOR HOURLY, 7-DAY FORECAST

| Type | Variable | Notes |
|------|----------|-------|
| Meteorological | Air Temp, Solar Rad, Wind, RH, Precip, Cloud Cover | Hourly forecasted |
| Hydrological | Discharge, Water Temp (lagged), Upstream Temp, Depth | Hourly or interpolated |
| Temporal | Hour, Day of Year, Lagged Inputs | For seasonality and inertia |

---

Want help building a data pipeline or setting up a neural network or regression model for this? Also happy to suggest feature engineering ideas or ML frameworks.

even more yapping...

Alright, let's go full send. Here's **everything you need to build a robust, hourly-resolution, 7-day forecast model for river water temperature**. I'll break it into the essential stages:

---

## 🧱 1. **Problem Setup**

**Goal**: Forecast hourly river water temperature for 168 hours (7 days) into the future
**Resolution**: Hourly
**Forecast Horizon**: 168 steps ahead
**Approach**: Data-driven (ML or hybrid), possibly with autoregressive structure

---

## 🗃️ 2. **Data Inputs (Features)**

### ⛅ **Meteorological (from forecast models or stations)**
| Feature               | Notes |
|-----------------------|-------|
| `air_temp` (°C)       | Most important; also use `lag_1h`, `lag_6h`, `lag_24h` |
| `solar_radiation` (W/m²) | Critical for diurnal heating |
| `wind_speed` (m/s)    | For surface heat exchange |
| `relative_humidity` (%) | Controls latent heat flux |
| `precipitation` (mm/h) | Use both `instantaneous` and `cumulative_6h` |
| `cloud_cover` (%)     | Proxy for solar input reduction |
| `dew_point` (optional) | Can help refine RH-based cooling |
| `shortwave_down` / `longwave_down` (optional) | For full surface energy budget |

---

### 🌊 **Hydrological (from gauge data or models)**
| Feature               | Notes |
|-----------------------|-------|
| `discharge` (m³/s)    | Modulates thermal inertia |
| `water_level` / `depth` | Controls storage + solar absorption |
| `water_temp_lag_1h` / `6h` / `24h` | Previous temps = crucial |
| `upstream_water_temp` (if available) | Lagged or real-time |
| `baseflow_index` or `groundwater_inflow` | If available; stabilizes temp |
| `velocity` (m/s)      | Optional, but helpful if slow-flow river |
| `turbidity` (NTU)     | Optional, impacts heat absorption in water column |

---

### 🧠 **Time-based (engineered features)**
| Feature               | Notes |
|-----------------------|-------|
| `hour_of_day` (0–23)  | Captures diurnal cycle |
| `day_of_week` (0–6)   | Optional for anthropogenic rivers |
| `day_of_year` (1–365) | Seasonal pattern |
| `sine_cos_time`       | Encode time using sine/cosine for smooth transitions |
| `is_weekend` / `holiday_flag` | Optional for dam-regulated systems |
| `prev_delta_temp`     | Change in temp over last 3–6h |

---

## 🏗️ 3. **Feature Engineering**

Here’s where you can really boost performance:

- **Lag features**: Create time-lagged versions of key inputs (temp, discharge, precip).
- **Rolling windows**: 3h, 6h, 12h rolling averages of variables like `air_temp`, `solar_rad`, `discharge`.
- **Cumulative sums**: Total precipitation in past 3h/6h.
- **Interaction terms**: E.g., `air_temp * solar_rad`, `wind_speed * RH`.

---

## 🧠 4. **Modeling Approaches**

### 💡 **Option A: Classic Machine Learning**
Great for interpretable, fast training.

- Random Forest
- Gradient Boosting (e.g. XGBoost, LightGBM)
- Support Vector Regression

✅ Pros: Interpretable, handles nonlinearity
❌ Cons: Not naturally autoregressive, harder to extrapolate

---

### 🔁 **Option B: Autoregressive + ML Hybrid**
Use past water temps as features to extend into future.

- Train model on:
  - `[X_t, y_{t-1}] → y_t`
  - Recursive: feed predicted `y_t` back as `y_{t+1}` input
- Optionally use **direct multi-output** models (`MultiOutputRegressor` or `seq2seq`)

✅ Good for short-horizon, can incorporate feedback
❌ Errors can compound over time

---

### 🧠 **Option C: Deep Learning**
Powerful for sequential, multi-step learning.

- LSTM or GRU (Recurrent NN)
- Temporal Convolutional Networks (TCN)
- Transformer models (like Informer or Temporal Fusion Transformer)
- N-BEATS or DLinear (SOTA for time series)

Use inputs shaped as:
`[batch, sequence_length, features] → forecast next 168 hourly steps`

✅ Pros: Captures temporal dependencies, learns sequence patterns
❌ Cons: Needs more data and tuning

---

## 🔍 5. **Training Strategy**

- **Train/test split**: Use time-based split (no leakage!).
- **Cross-validation**: Use walk-forward validation.
- **Loss functions**:
  - `MAE` or `RMSE` for regression
  - `Quantile Loss` if doing probabilistic forecasting

---

## 🧪 6. **Evaluation Metrics**

| Metric | Why it’s useful |
|--------|-----------------|
| MAE / RMSE | Standard error measures |
| NSE (Nash–Sutcliffe) | Good for hydrologic models |
| Bias | Systematic over/underprediction |
| Peak Timing Error | Critical for short-term forecasting |
| Diurnal Amplitude Error | For hourly resolution validation |
| Lead Time Accuracy | How performance drops with time horizon |

---

## 📊 7. **Visualization Ideas**

- **Time series plots** of forecast vs observed
- **Error plots by lead time** (e.g., MAE at t+1h, t+6h… t+168h)
- **Diurnal error plots** (average error by hour of day)
- **Heatmaps** of forecast skill by day/time
- **Feature importance (SHAP / permutation)** if using tree models

---

## 🛠️ 8. **Optional Enhancements**

- Ensemble multiple models (e.g., blend tree + neural net)
- Add **physical constraints** (e.g., no negative temps)
- **Data assimilation**: Blend observations with forecasts during runtime
- Train different models by season (summer/winter split)
- Use weather forecast ensemble inputs (to get uncertainty ranges)

---

If you let me know:
- Your location/river
- Available data types
- Preferred tech stack (e.g., Python with scikit-learn, PyTorch, TensorFlow, etc.)

…I can help you build the actual code scaffolding to get this off the ground. Ready to roll when you are.


## Claude 3.7 Sonnet

For forecasting river water temperature, I would suggest the following key hydrological and meteorological variables:

**Meteorological Variables:**
1. Air temperature (especially daily mean, minimum, and maximum)
2. Solar radiation/sunshine duration
3. Cloud cover
4. Wind speed and direction
5. Humidity/vapor pressure
6. Precipitation (amount, intensity, and type)

**Hydrological Variables:**
1. River discharge/flow rate
2. Upstream water temperature
3. Groundwater input and temperature
4. Water depth/river stage
5. Residence time
6. Thermal stratification (for deeper rivers)
7. Tributary inputs and their temperatures

**Additional Important Variables:**
1. Shading from riparian vegetation
2. Channel morphology (width, depth, slope)
3. Substrate type
4. Reservoir releases (if applicable)
5. Urban heat island effects (if in urban areas)
6. Snowmelt contribution

Air temperature typically has the strongest correlation with water temperature, but the combination of multiple variables will provide the most accurate forecast. Time lags should also be considered, as water temperature responds to meteorological changes with varying delays depending on river characteristics.

Would you like more detailed information about any of these variables or how they specifically influence river water temperature?

I'd be happy to explain why each of these variables is significant for river water temperature forecasting:

**Meteorological Variables:**

1. **Air temperature**: The primary driver of water temperature through heat exchange at the air-water interface. Rivers tend to equilibrate toward air temperature, though with lag time and dampened extremes. The relationship is typically strong but non-linear at extreme temperatures.

2. **Solar radiation**: Provides direct heating through shortwave radiation absorption. This is often the largest heat input during clear days and significantly impacts diurnal temperature patterns. Seasonal variations in day length and sun angle also influence long-term temperature regimes.

3. **Cloud cover**: Reduces incoming solar radiation and also affects longwave radiation exchange by reflecting heat back to the river surface. Acts as a natural buffer against temperature extremes.

4. **Wind speed/direction**: Affects evaporative cooling and sensible heat transfer rates. Higher wind speeds generally accelerate heat exchange between air and water, making temperature changes more rapid. Wind direction can influence fetch and mixing.

5. **Humidity/vapor pressure**: Controls evaporation rates, which represent a significant cooling mechanism. Lower humidity increases evaporative cooling, while high humidity reduces this effect, potentially allowing rivers to retain more heat.

6. **Precipitation**: Directly inputs water of different temperature, altering river temperature through thermal mixing. Also affects river volume, turbidity, and runoff patterns that influence thermal regimes.

**Hydrological Variables:**

1. **River discharge/flow rate**: Higher flows have greater thermal inertia and slower response to air temperature changes. Flow velocity affects residence time and exposure to atmospheric conditions.

2. **Upstream water temperature**: Provides the baseline condition that will be modified by local factors. Essential for connected river networks as upstream conditions propagate downstream.

3. **Groundwater input**: Often provides temperature-stabilizing effects as groundwater typically maintains consistent temperatures year-round (approximately mean annual air temperature). Springs can create thermal refugia in summer and winter.

4. **Water depth/river stage**: Deeper water bodies have greater thermal inertia and potential for vertical stratification. Shallow water responds more quickly to atmospheric conditions.

5. **Residence time**: Longer residence times allow greater equilibration with atmospheric conditions, while shorter times preserve upstream thermal signatures.

6. **Thermal stratification**: In deeper rivers, vertical temperature gradients can develop, affecting overall thermal dynamics and habitat conditions. Stratification can isolate bottom waters from atmospheric heating.

7. **Tributary inputs**: Introduce water of different temperatures, creating thermal heterogeneity and mixing zones. Important for heat budgets in larger river systems.

**Additional Variables:**

1. **Riparian shading**: Reduces direct solar radiation, potentially cooling water by several degrees. Seasonal leaf emergence/fall creates temporal patterns in shading effects.

2. **Channel morphology**: Width/depth ratios affect the surface area exposed to atmospheric conditions relative to volume. Steeper gradients increase turbulence and mixing.

3. **Substrate type**: Different materials (bedrock, gravel, etc.) have varying thermal properties affecting heat exchange with the riverbed. Dark substrates may absorb more heat.

4. **Reservoir releases**: Can completely alter natural thermal regimes through release of water from different reservoir depths, often creating colder summer and warmer winter conditions downstream.

5. **Urban heat island**: Urban areas are typically warmer than surrounding areas, heating urban streams through warmer air, runoff from heated surfaces, and reduced shading.

6. **Snowmelt contribution**: Provides consistently cold water inputs during melting periods, creating seasonal patterns in river thermal regimes in alpine or northern regions.

These variables interact in complex ways, with their relative importance varying by river type, season, and geographic region. Statistical and physically-based models typically incorporate multiple variables to capture these complex relationships.

# Next up

1. Try some simple autoregressive models using just air temperature in addition to the lagged water temperature and time features for seasonality.
2. Create a system to easily train and evaluate models with different sub-sets of features. In the end, I want to be able to know exactly which features in which model type performs best.

### Random note on evaluation

- Show the average error and its variance for each lag if possible. Would be interesting to see if there are hotspots and or how fast accuracy declines further into the future


